In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv")\
  .option("header", "true")\
  .option("inferSchema", "true")\
  .option("mode","PERMISSIVE")\
  .option("pathGlobFilter", "*.csv")\
  .load("/Volumes/prac/default/raw_data/amazon_sales_dataset.csv")

In [0]:
df.printSchema()
df.show(5)
df.count()

In [0]:
df.filter(df['customer_region'].isNull()).count()

In [0]:
from pyspark.sql import functions as F

# Calculate the expected revenue
df_check = df.withColumn("expected_revenue", F.col("discounted_price") * F.col("quantity_sold"))

# Filter for discrepancies (using a small threshold for floating point errors)
mismatched_orders = df_check.filter(F.abs(F.col("total_revenue") - F.col("expected_revenue")) > 0.01)

mismatched_orders.select("order_id", "total_revenue", "expected_revenue").show()

In [0]:
top_regions = df.filter(df["rating"] > 4.5) \
    .groupBy("customer_region") \
    .agg(F.sum("total_revenue").alias("high_rated_revenue")) \
    .orderBy(F.desc("high_rated_revenue"))

top_regions.show()

In [0]:
# Extract month from order_date
df_with_month = df.withColumn("order_month", F.month("order_date"))

discount_trend = df_with_month.groupBy("order_month", "product_category") \
    .agg(F.avg("discount_percent").alias("avg_discount")) \
    .orderBy("order_month", "avg_discount")

discount_trend.show()